# Multi-Source Sequence Fetch

This notebook demonstrates `run_sequence_fetch`, a thin orchestrator that retrieves biological sequences and structures from multiple public databases in a single call. It chains NCBI, UniProt, and PDB lookups with molecule-type routing and cross-fetcher ID sharing, so a UniProt ID resolved during protein retrieval is automatically reused when looking up a PDB structure. The notebook covers fetching a protein sequence by name, combining multiple molecule types in one request, processing batch queries across several targets, and bypassing name resolution by providing known accession identifiers directly.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("sequence_fetch")
display_overview("sequence_fetch")
display_docs_section("sequence_fetch", "Background")

# Unified Sequence Fetch

> `sequence-fetch` retrieves DNA, RNA, protein, and structure data for named targets across [NCBI](https://www.ncbi.nlm.nih.gov/) Entrez, [UniProt](https://www.uniprot.org/), and [PDB](https://www.rcsb.org/). It supports ID-first resolution, name+organism fallback, and strict molecular type checks.

## Background

**What does this tool retrieve?**
It fetches canonical molecular records from public databases: genomic DNA (`dna_genomic`), coding DNA (`dna_cds`), transcript RNA (`rna_transcript`), inferred pre-mRNA (`rna_premrna`), proteins (`protein`), and PDB structures (`structure`).

**Why is this important?**

* Reproducible sequence retrieval with explicit accession provenance
* ID-aware routing across NCBI, UniProt, and PDB in one interface
* Type-safe validation to avoid invalid requests (for example, protein for ncRNA)
* Batch-friendly results with per-request status and error reporting

**Scientific foundation:**
The tool wraps source APIs rather than using an internal predictive model. NCBI retrieval uses Entrez E-utilities (`esearch`, `efetch`), protein-centric retrieval prefers UniProt when IDs are present, and structure retrieval uses RCSB PDB entry endpoints. Resolution is deterministic and priority-based: provided IDs first, then name+organism fallback.

## Available tools

In [2]:
display_available_tools("sequence_fetch")

- **`run_sequence_fetch()`** — Fetch DNA, RNA, protein, and structure records from NCBI, UniProt, and PDB

### `run_sequence_fetch`

`run_sequence_fetch` accepts a list of `SequenceFetchRequest` objects, each specifying a target gene or protein name, an organism for disambiguation, and the desired molecule types. It searches UniProt, NCBI, and PDB in sequence, sharing resolved IDs across fetchers so each database is queried at most once per request. Results are returned as a `SequenceFetchOutput` with one `SequenceFetchResult` per request, containing the fetched sequences, structures, resolved IDs, and any per-type errors.

In [3]:
from proto_tools import SequenceFetchInput, SequenceFetchConfig, SequenceFetchRequest, run_sequence_fetch

In [4]:
# Display docs
display_api_reference("sequence_fetch", "input", "run_sequence_fetch")

# Fetch the lac repressor protein sequence from Escherichia coli
inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        )
    ]
)

**Input** — `SequenceFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `requests` | `List[SequenceFetchRequest]` | required | One or more retrieval requests. |
| `request_id` | `string` |  | Optional caller-provided request identifier. |
| `target_name` | `string` | required | Gene, protein, or RNA name to resolve. |
| `organism` | `string` | required | Organism name used for disambiguation. |
| `sequence_types` | `List[string]` | required | Requested outputs: protein, dna, rna, or structure. |
| `uniprot_id` | `string` |  | UniProt accession override. |
| `genbank_accession` | `string` |  | GenBank accession override. |
| `refseq_accession` | `string` |  | RefSeq accession override. |
| `pdb_id` | `string` |  | PDB accession override. |
| `gene_id` | `string` |  | NCBI Gene ID override. |
| `protein_id` | `string` |  | NCBI protein accession override. |
| `transcript_id` | `string` |  | Transcript accession override. |
| `genomic_coordinates` | `string` |  | Genomic interval like NC\_000913.3:1-100:+. |
| `additional_ids` | `Dict[string, string]` |  | Extra IDs used for custom routing. |

In [5]:
# Display config docs
display_api_reference("sequence_fetch", "config", "run_sequence_fetch")

# Use default config | see docs above for all fields
config = SequenceFetchConfig()

**Config** — `SequenceFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `max_candidates_per_source` | `integer` | `5` | Maximum database candidates to evaluate per name-based search. |
| `ncbi_api_key` | `string` |  | Optional API key for NCBI Entrez. |
| `ncbi_email` | `string` |  | Optional contact email for NCBI requests. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the fetch
output = run_sequence_fetch(inputs, config)

In [7]:
# Display output docs
display_api_reference("sequence_fetch", "output", "run_sequence_fetch")

# Inspect the fetched protein sequence
result = output.results[0]
print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Type: {seq.sequence_type}")
print(f"Source: {seq.source_database} ({seq.accession})")
print(f"Length: {seq.length}")
print(f"Preview: {seq.sequence[:60]}...")

**Output** — `SequenceFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[SequenceFetchResult]` |  | Per-request retrieval outcomes. |
| `request_id` | `string` | required | Request identifier used in this result. |
| `target_name` | `string` | required | Original target name. |
| `organism` | `string` | required | Original organism name. |
| `requested_types` | `List[string]` | required | Requested output molecule types. |
| `status` | `enum` | required | One of success, warning, or failed. |
| `fetched_sequences` | `List[FetchedSequence]` |  | Retrieved sequence records. |
| `fetched_structures` | `List[FetchedStructure]` |  | Retrieved structure records. |
| `resolved_ids` | `Dict[string, string]` |  | IDs resolved or used during retrieval. |
| `warnings` | `List[string]` |  | Non-fatal warnings. |
| `errors` | `List[string]` |  | Fatal or partial failure messages. |

Target: lacI (Escherichia coli)
Status: success
Resolved IDs: {'uniprot_id': 'A0A094XSW9'}

Type: protein
Source: uniprot (A0A094XSW9)
Length: 319
Preview: MAELNYIPNRVAQQLAGKQSLLIGVATSSLALHAPSQIVAAIKSRADQLGASVVVSMVER...


#### Multi-Type Request

A single request can fetch multiple molecule types at once. Here we retrieve the protein sequence, coding DNA sequence, and an experimentally determined structure for TP53. The orchestrator resolves IDs once and shares them across fetchers, so a UniProt ID found during protein retrieval is reused when looking up the associated PDB structure.

In [8]:
# Fetch protein, CDS, and structure for TP53 in a single request
multi_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "dna_cds", "structure"],
        )
    ]
)

multi_output = run_sequence_fetch(multi_inputs, config)
result = multi_output.results[0]

print(f"Target: {result.target_name} ({result.organism})")
print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

for seq in result.fetched_sequences:
    preview = seq.sequence[:50] + ("..." if len(seq.sequence) > 50 else "")
    print(f"  {seq.sequence_type}: {seq.source_database} ({seq.accession}), length={seq.length}")
    print(f"    {preview}")

for struct in result.fetched_structures:
    res_str = f"{struct.resolution:.1f} A" if struct.resolution else "N/A"
    print(f"  structure: {struct.pdb_id}, {struct.method}, {res_str}")
    print(f"    {struct.title}")

Target: TP53 (Homo sapiens)
Status: warning
Resolved IDs: {'uniprot_id': 'P04637', 'cds_accession': 'PZ086170.1_cds_YDH27063.1_1', 'pdb_id': '1A1U'}

  protein: uniprot (P04637), length=393
    MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDI...
  dna_cds: ncbi (PZ086170.1_cds_YDH27063.1_1), length=123
    TGGGTTGATTCCACACCCCCGCCCGGCACCCGCGTCCGCGCCGTGGCCAT...
  structure: 1A1U, SOLUTION NMR, N/A
    SOLUTION STRUCTURE DETERMINATION OF A P53 MUTANT DIMERIZATION DOMAIN, NMR, MINIMIZED AVERAGE STRUCTURE


#### Batch Requests

Multiple targets can be processed in a single call by including multiple `SequenceFetchRequest` objects. Each request is resolved independently against the upstream databases.

In [9]:
# Fetch three proteins in one call
batch_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="lacI",
            organism="Escherichia coli",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="insulin",
            organism="Homo sapiens",
            sequence_types=["protein"],
        ),
        SequenceFetchRequest(
            target_name="GFP",
            organism="Aequorea victoria",
            sequence_types=["protein"],
        ),
    ]
)

batch_output = run_sequence_fetch(batch_inputs, config)

print(f"Completed: {batch_output.num_completed}/{batch_output.num_requests}")
print()

for result in batch_output.results:
    if result.fetched_sequences:
        seq = result.fetched_sequences[0]
        print(f"{result.target_name}: {seq.source_database} ({seq.accession}), length={seq.length}")
    else:
        print(f"{result.target_name}: {result.status} -- {result.errors}")

Completed: 3/3

lacI: uniprot (A0A094XSW9), length=319
insulin: uniprot (P01308), length=110
GFP: uniprot (A0A059PIQ0), length=251


#### Direct ID Override

When you already have a UniProt accession or PDB ID, you can provide it directly to skip the name search step. This is faster and avoids any ambiguity from name resolution.

In [10]:
# Use known UniProt and PDB IDs to skip name resolution
direct_inputs = SequenceFetchInput(
    requests=[
        SequenceFetchRequest(
            target_name="TP53",
            organism="Homo sapiens",
            sequence_types=["protein", "structure"],
            uniprot_id="P04637",
            pdb_id="1TSR",
        )
    ]
)

direct_output = run_sequence_fetch(direct_inputs, config)
result = direct_output.results[0]

print(f"Status: {result.status}")
print(f"Resolved IDs: {result.resolved_ids}")
print()

seq = result.fetched_sequences[0]
print(f"Protein: {seq.accession}, length={seq.length}")

struct = result.fetched_structures[0]
print(f"Structure: {struct.pdb_id}, {struct.method}")

Status: success
Resolved IDs: {'uniprot_id': 'P04637', 'pdb_id': '1TSR'}

Protein: P04637, length=393
Structure: 1TSR, X-RAY DIFFRACTION


## Export Results

Fetched sequences can be saved to FASTA or JSON format for downstream analysis. The FASTA export includes structured headers with target name, sequence type, source database, and accession for each record.

In [11]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("sequence_fetch_results", export_path=output_dir, file_format="fasta")
print(f"Exported to {output_dir / 'sequence_fetch_results.fasta'}")

output.export("sequence_fetch_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'sequence_fetch_results.json'}")

Exported to example_output/sequence_fetch_results.fasta
Exported to example_output/sequence_fetch_results.json
